# Maia Lite 0.3.0 — Provenance Enrichment v7

**Purpose:** enrich the 20 v6 source entries with evidence, disaggregate Wikipedia/Wikibooks JSONL into document-level records, and generate a human-review table. **No automatic rights approval or training export.**

Inputs: `corpus_v6/reports/source_rights_review_v6.json`, `corpus_v5/reports/maia_corpus_v030_manifest_v5.json` and source files. Outputs under `corpus_v7/reports/` and `corpus_v7/documents/`. Re-running preserves all human review fields. It does not modify v6 or its exports.

For Wikimedia content, record page-level revision IDs if present; otherwise mark missing rather than fabricate them. For Gutenberg, use the canonical ebook link and parse bibliographic headers locally; a canonical link is **not** a verified license or edition. Official source policies: https://www.gutenberg.org/policy/license and https://www.mediawiki.org/wiki/Wikimedia_APIs/Rate_limits.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
from urllib.parse import quote
import hashlib, json, csv, re, unicodedata, time
import pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Outside Colab: Drive mount skipped')
ROOT=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat')
V6=ROOT/'corpus_v6/reports/source_rights_review_v6.json'
V5=ROOT/'corpus_v5/reports/maia_corpus_v030_manifest_v5.json'
OUT=ROOT/'corpus_v7'; REPORT=OUT/'reports'; DOCS=OUT/'documents'
for p in (REPORT,DOCS):p.mkdir(parents=True,exist_ok=True)
assert V6.exists(), f'Missing v6 ledger: {V6}'
assert V5.exists(), f'Missing v5 manifest: {V5}'
reviews=json.loads(V6.read_text(encoding='utf-8'))
manifest=json.loads(V5.read_text(encoding='utf-8'))
source_map={x['source']:x for x in manifest.get('sources',[]) if x.get('source')}
print('v6 entries:',len(reviews),'v5 entries:',len(source_map))

## 1. Enrich from existing local files (offline by default)

This phase is intentionally network-free. Gutenberg header parsing is heuristic; extracted author, translator, title and rights notices are **candidates for human verification**. Wikimedia article identity is parsed from JSONL fields; never assume an article revision if not recorded. JSONL rows without identity receive stable local IDs and are flagged for verification.

In [ ]:
def sha256_file(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()

def norm(x):return unicodedata.normalize('NFC',str(x)).strip()
def pick(d,*keys):
    for k in keys:
        v=d.get(k)
        if v is not None and str(v).strip():return v
    return None

def gutenberg_header(text):
    head=text[:16000]
    fields={}
    for label,pattern in {
        'title':r'(?im)^Title:\s*(.+)$',
        'author':r'(?im)^Author:\s*(.+)$',
        'translator':r'(?im)^Translator:\s*(.+)$',
        'illustrator':r'(?im)^Illustrator:\s*(.+)$',
        'release_date':r'(?im)^Release Date:\s*(.+)$',
        'language_declared':r'(?im)^Language:\s*(.+)$',
        'copyright_notice':r'(?im)^Copyright[^\n]{0,100}$',
    }.items():
        match=re.search(pattern,head)
        fields[label]=match.group(1).strip() if match and match.lastindex else (match.group(0).strip() if match else None)
    fields['permission_marker_present']=bool(re.search(r'(?i)posted with (?:the )?permission|copyright holder',head))
    return fields

def parse_jsonl(p):
    with p.open('r',encoding='utf-8',errors='replace') as f:
        for i,line in enumerate(f,1):
            if not line.strip():continue
            try: row=json.loads(line)
            except json.JSONDecodeError:
                yield i, {'_parse_error':'invalid JSON'};continue
            if isinstance(row,str):row={'text':row}
            if not isinstance(row,dict):row={'_parse_error':'unexpected JSON type'}
            yield i,row

def page_record(parent,lang,domain,kind,row,line_number):
    title=pick(row,'title','page_title','pageTitle','name')
    pageid=pick(row,'pageid','page_id','pageId')
    revid=pick(row,'revid','rev_id','revision_id','revisionId','lastrevid')
    rev= row.get('revision')
    if isinstance(rev,dict):revid=revid or pick(rev,'revid','id')
    url=pick(row,'url','fullurl','canonical_url','source_url')
    if not url and title:
        base={'wikipedia':'wikipedia.org','wikibooks':'wikibooks.org'}[kind]
        url=f'https://{lang}.{base}/wiki/{quote(str(title).replace(" ","_"),safe="/():")}'
    content=pick(row,'text','content','extract','body')
    identity=f'{kind}:{lang}:pageid:{pageid}' if pageid else (f'{kind}:{lang}:title:{title}' if title else f'{parent}:line:{line_number}')
    key=hashlib.sha256(identity.encode()).hexdigest()[:20]
    return {'document_id':f'{kind}-{lang}-{key}','parent_source':parent,'lang':lang,'domain':domain,
            'kind':kind,'title':title,'page_id':pageid,'revision_id':revid,'source_url':url,
            'line_number':line_number,'content_chars':len(str(content)) if content else 0,
            'has_content':bool(content),'identity_status':'verified_fields_present' if (title or pageid) else 'missing_page_identity',
            'revision_status':'present_unverified' if revid else 'missing_revision',
            'declared_license':pick(row,'license','license_url','license_identifier'),
            'raw_metadata_keys':sorted(row.keys()),'local_file_sha256':None}

In [ ]:
# 2. Build evidence and document-level index. No requests to external websites.
source_evidence={};documents=[];issues=[]
for source,review in sorted(reviews.items()):
    parts=Path(source).parts
    if len(parts)<3:issues.append({'source':source,'issue':'invalid source key'});continue
    lang,domain=parts[0],parts[1]
    candidate=source_map.get(source,{})
    p=Path(candidate.get('source_path','')) if candidate.get('source_path') else None
    if p is None or not p.is_file():
        # v6 imported files can have a different relative path
        p=ROOT/'corpus_v6'/source if source.startswith('imports/') else None
    if p is None or not p.is_file():
        issues.append({'source':source,'issue':'file not found'});continue
    sha=sha256_file(p)
    if 'gutenberg' in parts:
        m=re.search(r'(\d+)\.txt$',p.name)
        ebook_id=m.group(1) if m else None
        meta=gutenberg_header(p.read_text(encoding='utf-8',errors='replace'))
        url=f'https://www.gutenberg.org/ebooks/{ebook_id}' if ebook_id else None
        ev={'kind':'gutenberg','source':source,'local_path':str(p),'local_sha256':sha,
            'canonical_url':url,'ebook_id':ebook_id,'bibliography_candidate':meta,
            'rights_reference':'https://www.gutenberg.org/policy/license',
            'rights_evidence_status':'reference_only_not_verified_for_edition',
            'rights_review_required':['jurisdiction','edition','translator','rights notice','trademark/redistribution']}
        source_evidence[source]=ev
        documents.append({'document_id':f'gutenberg-{ebook_id or sha[:16]}','parent_source':source,
            'lang':lang,'domain':domain,'kind':'gutenberg','title':meta.get('title'),
            'page_id':None,'revision_id':None,'source_url':url,'line_number':None,
            'content_chars':p.stat().st_size,'has_content':True,
            'identity_status':'ebook_id_from_filename','revision_status':'not_applicable',
            'declared_license':None,'raw_metadata_keys':[], 'local_file_sha256':sha})
    elif p.suffix=='.jsonl' and ('wikipedia' in parts or 'wikibooks' in parts):
        kind='wikibooks' if 'wikibooks' in parts else 'wikipedia'
        source_evidence[source]={'kind':kind,'source':source,'local_path':str(p),
            'local_sha256':sha,'project_url':f'https://{lang}.{kind}.org/',
            'rights_reference':f'https://{lang}.{kind}.org/wiki/Project:Copyrights',
            'rights_evidence_status':'project_reference_only_article_exceptions_unreviewed'}
        for lineno,row in parse_jsonl(p):
            if '_parse_error' in row:
                issues.append({'source':source,'line':lineno,'issue':row['_parse_error']});continue
            doc=page_record(source,lang,domain,kind,row,lineno)
            doc['local_file_sha256']=sha
            documents.append(doc)
    else:
        source_evidence[source]={'kind':'other','source':source,'local_path':str(p),
            'local_sha256':sha,'rights_evidence_status':'manual_review_required'}
        documents.append({'document_id':f'local-{sha[:20]}','parent_source':source,
            'lang':lang,'domain':domain,'kind':'other','title':None,'page_id':None,
            'revision_id':None,'source_url':None,'line_number':None,
            'content_chars':p.stat().st_size,'has_content':True,'identity_status':'local_file_only',
            'revision_status':'not_applicable','declared_license':None,
            'raw_metadata_keys':[],'local_file_sha256':sha})
print('Sources with evidence:',len(source_evidence),'Document records:',len(documents),'Issues:',len(issues))
print('Kinds:',dict(Counter(d['kind'] for d in documents)))

## 2. Merge evidence into review ledger without overwriting human decisions

`source_rights_review_v7.json` retains the v6 human fields exactly and adds `automated_evidence`. **Never** set `approved_for_training` or `review_complete` from metadata. Review at document level separately for Wikimedia content; approval of the parent collection does not automatically authorize every article.

In [ ]:
# 3. Preserve prior v7 reviews, if present, including human edits.
LEDGER=REPORT/'source_rights_review_v7.json'
previous=json.loads(LEDGER.read_text(encoding='utf-8')) if LEDGER.exists() else {}
merged={}
for key,base in reviews.items():
    entry=dict(base)
    entry.update(previous.get(key,{}))
    entry['automated_evidence']=source_evidence.get(key,{})
    merged[key]=entry
LEDGER.write_text(json.dumps(merged,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
print('Source review ledger:',LEDGER)
print('Human decisions preserved:',sum(bool(previous.get(k)) for k in merged))

In [ ]:
# 4. Document-level review ledger, one entry per Wikipedia/Wikibooks article
DOC_LEDGER=REPORT/'document_rights_review_v7.json'
prior_docs=json.loads(DOC_LEDGER.read_text(encoding='utf-8')) if DOC_LEDGER.exists() else {}
doc_reviews={}
for d in documents:
    k=d['document_id']
    if k in doc_reviews:
        issues.append({'source':d['parent_source'],'issue':'document ID collision','document_id':k})
        continue
    existing=prior_docs.get(k,{})
    doc_reviews[k]={
        'document_id':k,'parent_source':d['parent_source'],
        'status':existing.get('status','pending_review'),
        'review_complete':existing.get('review_complete',False),
        'approved_for_training':existing.get('approved_for_training',False),
        'reviewer':existing.get('reviewer',''),
        'reviewed_utc':existing.get('reviewed_utc',''),
        'rights_basis':existing.get('rights_basis',''),
        'license_identifier':existing.get('license_identifier',''),
        'attribution':existing.get('attribution',''),
        'obligations':existing.get('obligations',''),
        'training_use_checked':existing.get('training_use_checked',False),
        'redistribution_checked':existing.get('redistribution_checked',False),
        'translation_edition_checked':existing.get('translation_edition_checked',False),
        'notes':existing.get('notes',''),
        'automated_evidence':{k2:v for k2,v in d.items() if k2 not in ('raw_metadata_keys',)}
    }
DOC_LEDGER.write_text(json.dumps(doc_reviews,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
print('Document review ledger:',DOC_LEDGER,'records:',len(doc_reviews))

## 3. Review queue and source-level independence

A file containing many Wikipedia articles is **not** a single independent document. This notebook creates an article-level inventory and a candidate source-family key. It does not assign train/validation/test splits yet; article revisions, duplicates and linked/translated articles require further grouping before splitting. No raw text is copied into a training export.

In [ ]:
# 5. Export audit tables and machine-readable inventory
frame=pd.DataFrame(documents)
if frame.empty:raise RuntimeError('No documents found. Check source paths.')
frame['needs_revision']=frame['revision_status'].eq('missing_revision')
frame['needs_identity']=frame['identity_status'].eq('missing_page_identity')
frame['rights_status']=[doc_reviews[d]['status'] for d in frame['document_id']]
frame['source_family']=[f"{d['kind']}:{d['lang']}:{d['page_id'] or d['title'] or d['document_id']}" for d in documents]
frame.to_csv(REPORT/'document_inventory_v7.csv',index=False)
frame.to_json(REPORT/'document_inventory_v7.jsonl',orient='records',lines=True,force_ascii=False)
source_rows=[]
for src,ev in source_evidence.items():
    source_rows.append({'source':src,'kind':ev['kind'],'canonical_url':ev.get('canonical_url',ev.get('project_url','')),
                        'sha256':ev.get('local_sha256'),'review_status':merged[src].get('status'),
                        'documents':sum(d['parent_source']==src for d in documents),
                        'rights_reference':ev.get('rights_reference','')})
pd.DataFrame(source_rows).to_csv(REPORT/'source_review_queue_v7.csv',index=False)
pd.DataFrame(issues,columns=['source','issue','line','document_id']).to_csv(REPORT/'extraction_issues_v7.csv',index=False)
print(frame.groupby(['lang','kind'],dropna=False).agg(documents=('document_id','size'),
      missing_revisions=('needs_revision','sum'),missing_identity=('needs_identity','sum')).to_string())
print('Review queue:',REPORT/'source_review_queue_v7.csv')

## 4. Optional Wikimedia metadata recovery (disabled by default)

Use only for **specific pages missing revisions** and after setting a real contact address. The code uses one request at a time, a descriptive User-Agent, `maxlag`, pauses and respects HTTP 429/503 `Retry-After`. It never circumvents throttling and does not auto-approve rights. **Do not run for bulk collections; prefer official dumps.**

In [ ]:
import requests, email.utils
from datetime import datetime,timezone
ENABLE_NETWORK=False
CONTACT_USER_AGENT='' # Example: 'MaiaCorpusResearch/0.3 (https://www.maiaplatform.org/; your-email@domain.org)'
MAX_REQUESTS=12

def fetch_page_info(lang,kind,title,session):
    url=f'https://{lang}.{kind}.org/w/api.php'
    params={'action':'query','format':'json','formatversion':2,'prop':'info|revisions',
            'rvprop':'ids|timestamp','titles':title,'inprop':'url','maxlag':5}
    for attempt in range(4):
        response=session.get(url,params=params,timeout=35)
        if response.status_code in (429,503):
            raw=response.headers.get('Retry-After','')
            try:delay=max(1,min(300,int(raw)))
            except ValueError:
                try:delay=max(1,min(300,(email.utils.parsedate_to_datetime(raw)-datetime.now(timezone.utc)).total_seconds()))
                except (TypeError,ValueError,OverflowError):delay=min(120,10*(2**attempt))
            print('Throttled:',response.status_code,'waiting',delay,'seconds')
            time.sleep(delay)
            continue
        response.raise_for_status()
        payload=response.json()
        if 'error' in payload:
            raise RuntimeError(str(payload['error']))
        return (payload.get('query') or {}).get('pages',[])
    raise RuntimeError('Retry budget exhausted')

if ENABLE_NETWORK:
    if not CONTACT_USER_AGENT or '@' not in CONTACT_USER_AGENT or 'https://' not in CONTACT_USER_AGENT:
        raise ValueError('Set a real User-Agent containing a contact URL and email')
    session=requests.Session();session.headers['User-Agent']=CONTACT_USER_AGENT
    recovered={}
    queue=[d for d in documents if d['kind'] in ('wikipedia','wikibooks')
           and d['title'] and not d['revision_id']]
    for d in queue[:MAX_REQUESTS]:
        try:
            pages=fetch_page_info(d['lang'],d['kind'],str(d['title']),session)
            if pages and not pages[0].get('missing'):
                p=pages[0];rev=(p.get('revisions') or [{}])[0]
                recovered[d['document_id']]={'pageid':p.get('pageid'),'revid':rev.get('revid'),
                                             'timestamp':rev.get('timestamp'),'fullurl':p.get('fullurl'),
                                             'fetched_utc':datetime.now(timezone.utc).isoformat()}
        except Exception as exc:print('Could not recover',d['document_id'],type(exc).__name__,str(exc)[:180])
        time.sleep(2)
    target=REPORT/'wikimedia_metadata_recovery_v7.json'
    old=json.loads(target.read_text(encoding='utf-8')) if target.exists() else {}
    old.update(recovered)
    target.write_text(json.dumps(old,ensure_ascii=False,indent=2),encoding='utf-8')
    print('Recovered:',len(recovered),'Saved:',target)
else:
    print('Network disabled. No API calls made.')

## 5. Final audit manifest

The notebook **does not export approved text**. It leaves production readiness false because document-level rights, quality, duplicate families, cross-language links, and benchmark leakage need verification. Approval must be recorded separately with evidence.

In [ ]:
report={
 'version':'v7','generated_utc':datetime.now(timezone.utc).isoformat(),
 'sources_v6':len(reviews),'sources_with_evidence':len(source_evidence),
 'documents_total':len(documents),'documents_by_kind':dict(Counter(d['kind'] for d in documents)),
 'documents_missing_revision':int(frame['needs_revision'].sum()),
 'documents_missing_identity':int(frame['needs_identity'].sum()),
 'document_reviews_pending':sum(x['status']=='pending_review' for x in doc_reviews.values()),
 'extraction_issues':len(issues),'training_export_created':False,'production_ready':False,
 'blockers':['Source-level and document-level rights review incomplete',
             'Revision/identity metadata may be absent or unverified',
             'Independent source-family deduplication and splitting not completed',
             'Benchmark leakage and multilingual quality audits not completed',
             'Final tokenizer not trained'],
 'outputs':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in REPORT.iterdir() if p.is_file()}
}
manifest_path=REPORT/'maia_corpus_v030_manifest_v7.json'
manifest_path.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
print('Manifest:',manifest_path)
print(json.dumps({k:v for k,v in report.items() if k!='outputs'},indent=2,ensure_ascii=False))

### Next step

Inspect `source_review_queue_v7.csv`, `document_inventory_v7.csv` and `extraction_issues_v7.csv`. Check a few representative Wikipedia and Wikibooks records for page ID/revision completeness, then manually review source and article rights. This is evidence gathering, **not** legal clearance or proof of training eligibility.